In [ ]:
import os
import requests
import json

from pathlib import Path
from datasets import load_dataset

import sys
sys.path.append("..")

In [ ]:
data_path = Path("../data")
ckpt_path = Path("../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)

warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [ ]:
len(warandpeace)

In [ ]:
chars = sorted(list(set(warandpeace)))
vocab_size = len(chars)

print("".join(chars))
print(vocab_size)

In [ ]:
tokenizer_scratchy_path = tokenizer_path / "scratchy.json"

with open(tokenizer_scratchy_path, 'w') as file:
    json.dump(chars, file)

with open(tokenizer_scratchy_path, 'r') as file:
    chars = json.load(file)

In [ ]:
from src.tokenizer import ScratchyTokenizer

In [ ]:
tokenizer = ScratchyTokenizer(chars)

In [ ]:
print(tokenizer.encode("привет как дела yoo"))
print(tokenizer.decode(tokenizer.encode("привет алоу, yoo")))

In [ ]:
# https://github.com/google/sentencepiece/blob/master/python/tokenizer_comparison_cheat_sheet.md

In [ ]:
# https://github.com/google/sentencepiece

In [ ]:
import sentencepiece as spm


sentencepiece_tokenizer_path = tokenizer_path / "sentencepiece"
spm.SentencePieceTrainer.train(
    input=warandpeace_path, 
    model_prefix=str(sentencepiece_tokenizer_path), 
    vocab_size=3000
)

In [ ]:

sp = spm.SentencePieceProcessor(
    model_file=str(sentencepiece_tokenizer_path.with_suffix(".model"))
)

In [ ]:
text = "привет как дела yoo, motherfuckerr"
pieces = sp.encode(text, return_type=str)
ids = sp.encode(text, return_type=int)

print(pieces)
print(ids)

sp.decode(pieces, return_type=str)

In [ ]:
# https://github.com/openai/tiktoken

In [ ]:
import tiktoken

# pretrained 
enc = tiktoken.get_encoding("o200k_base")

text = "привет как дела yoo, motherfucker"
encoded = enc.encode(text)
print(encoded)
print(enc.decode(encoded))


In [ ]:
from tiktoken._educational import SimpleBytePairEncoding

gpt2_pattern = (
        r"""'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )

enc = SimpleBytePairEncoding.train(warandpeace, vocab_size=600, pat_str=gpt2_pattern)


In [ ]:
import pickle

sentencepiece_tokenizer_path = tokenizer_path / "tiktoken_bpe.pkl"

with open(sentencepiece_tokenizer_path, 'wb') as f:
    pickle.dump(enc, f)

with open(sentencepiece_tokenizer_path, 'rb') as f:
    enc = pickle.load(f)

In [ ]:

text = "привет как дела yoo, motherfucker"
encoded = enc.encode(text)
print(encoded)
print(enc.decode(encoded))

In [ ]:
# https://github.com/huggingface/tokenizers

In [ ]:
from tokenizers import ByteLevelBPETokenizer, CharBPETokenizer

tokenizer = CharBPETokenizer()

tokenizer.train(str(warandpeace_path))

In [ ]:

text = "привет как дела yoo, motherfucker"
encoded = tokenizer.encode(text)

print(encoded.tokens)
print(encoded.ids)
print(tokenizer.decode(encoded.ids))

In [ ]:

sentencepiece_tokenizer_path = tokenizer_path / "huggingface_char_bpe"
sentencepiece_tokenizer_path.mkdir(exist_ok=True, parents=True)
tokenizer.save_model(str(sentencepiece_tokenizer_path))


tokenizer = CharBPETokenizer.from_file(
    str(sentencepiece_tokenizer_path / "vocab.json"),
    str(sentencepiece_tokenizer_path / "merges.txt")
)